In [1]:
import importlib.util
import subprocess
import sys

pacotes_obrigatorios = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib'
}

pacotes_faltando = [
    nome_pip
    for nome_modulo, nome_pip in pacotes_obrigatorios.items()
    if importlib.util.find_spec(nome_modulo) is None
]

if pacotes_faltando:
    print('Instalando dependencias que estao faltando:', ', '.join(pacotes_faltando))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pacotes_faltando])
else:
    print('Dependencias principais ja instaladas.')

Instalando dependencias que estao faltando: matplotlib


In [2]:
from pathlib import Path

import pandas as pd

for pasta_base in [Path.cwd(), Path.cwd().parent]:
    if (pasta_base / 'data' / 'train.csv').exists() and (pasta_base / 'data' / 'test.csv').exists():
        project_root = pasta_base
        break
else:
    raise FileNotFoundError('Não encontrei a pasta data com train.csv e test.csv. Rode o notebook dentro da pasta do projeto.')

data_dir = project_root / 'data'

# 1. Carregar os arquivos CSV
print("Carregando os CSVs...")
train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

# 2. Separa a variável alvo (Target) do dataset de treino
y_train = train_df['winner']

# Guardamos os nomes das seleções do teste para usar no CSV final
times_teste = test_df['team_name']

# 3. Removemos as colunas de texto que não são variáveis preditivas (Identificadores e Target)
# Não podemos enviar texto puro (strings) para o modelo do sklearn
features_train = train_df.drop(columns=['winner', 'team_name', 'country_code'])
features_test = test_df.drop(columns=['team_name', 'country_code'])

# 4. Concatenamos o Treino e Teste temporariamente
# Porque se o treino tiver uma confederação que não existe no teste, 
# o One-Hot Encoding vai criar um número diferente de colunas e o modelo vai se arrebentar.
# Juntamos os dois para garantir que o pd.get_dummies cria as colunas exatas para ambos.
df_combined = pd.concat([features_train, features_test])

# 5. One-Hot Encoding: Transformar a coluna 'confederation' (texto) em colunas binárias numéricas (0 ou 1)
df_combined = pd.get_dummies(df_combined, columns=['confederation'])

# 6. Separar novamente em Treino  e Teste 
# Usamos o tamanho original do train_df para fazer o corte (slicing) no exato ponto certo
X_train = df_combined.iloc[:len(train_df)].copy()
X_test_final = df_combined.iloc[len(train_df):].copy()

# 7. Verificação final de sanidade (Opcional, mas recomendado para garantir que não há dados nulos)
X_train.fillna(0, inplace=True)
X_test_final.fillna(0, inplace=True)

print("Pré-processamento concluído com sucesso!\n")
print(f"\nFormato final dos dados de treino: {X_train.shape}")
print(f"\nFormato final dos dados de teste: {X_test_final.shape}")

Carregando os CSVs...
Pré-processamento concluído com sucesso!


Formato final dos dados de treino: (1000, 28)

Formato final dos dados de teste: (250, 28)


## Task 3 - Previsões oficiais e Feature Importance

Esta etapa aplica o modelo treinado no `test.csv`, gera o ranking de probabilidades e lista as variáveis que mais influenciaram a decisão do modelo.

In [3]:
import sys
from pathlib import Path

pasta_src = project_root / 'src'
if not (pasta_src / 'train.py').exists():
    raise FileNotFoundError('Não encontrei src/train.py. Confira se o arquivo do treino está no projeto.')

sys.path.insert(0, str(pasta_src.resolve()))

from train import treinar_modelo_copa

# A Task 3 depende do modelo treinado pela Task 2.
# Usamos a função treinar_modelo_copa criada no arquivo src/train.py.
modelo = treinar_modelo_copa(X_train, y_train)

print("Modelo pronto para gerar as previsões oficiais.")

Iniciando a Task 2: Treinamento e Validação do Modelo...
Dados divididos: conjunto de treino e conjunto de validação criados.
Algoritmo Random Forest instanciado.
Modelo treinado com sucesso.

MÉTRICAS DE AVALIAÇÃO DO MODELO
Acurácia: 60.50%

Matriz de Confusão:
[[67 25]
 [54 54]]

Relatório de Classificação Completo:
              precision    recall  f1-score   support

           0       0.55      0.73      0.63        92
           1       0.68      0.50      0.58       108

    accuracy                           0.60       200
   macro avg       0.62      0.61      0.60       200
weighted avg       0.62      0.60      0.60       200

Modelo pronto para gerar as previsões oficiais.


In [4]:
from pathlib import Path

outputs_dir = project_root / 'outputs'
outputs_dir.mkdir(exist_ok=True)

# predict_proba retorna a probabilidade de cada classe.
# Como winner=1 representa campeão/vencedor, pegamos a coluna da classe 1.
classes_modelo = list(modelo.classes_)
classe_vencedor = 1 if 1 in classes_modelo else classes_modelo[-1]
indice_classe_vencedor = classes_modelo.index(classe_vencedor)
probabilidades_vitoria = modelo.predict_proba(X_test_final)[:, indice_classe_vencedor]

previsoes_test = pd.DataFrame({
    'team_name': times_teste,
    'country_code': test_df['country_code'],
    'confederation': test_df['confederation'],
    'probabilidade_campeao': probabilidades_vitoria,
    'probabilidade_campeao_pct': (probabilidades_vitoria * 100).round(2)
})

previsoes_detalhadas_path = outputs_dir / 'previsoes_detalhadas_test.csv'
previsoes_test.to_csv(previsoes_detalhadas_path, index=False, encoding='utf-8-sig')

# O test.csv possui seleções repetidas. Para o ranking oficial, agregamos por seleção
# usando a probabilidade média prevista pelo modelo.
ranking_probabilidades = (
    previsoes_test
    .groupby(['team_name', 'country_code', 'confederation'], as_index=False)
    .agg(
        probabilidade_campeao=('probabilidade_campeao', 'mean'),
        qtd_amostras=('probabilidade_campeao', 'size')
    )
)
ranking_probabilidades['probabilidade_campeao_pct'] = (
    ranking_probabilidades['probabilidade_campeao'] * 100
).round(2)

ranking_probabilidades = (
    ranking_probabilidades
    .sort_values(['probabilidade_campeao', 'team_name'], ascending=[False, True])
    .reset_index(drop=True)
)
ranking_probabilidades.insert(0, 'posicao', ranking_probabilidades.index + 1)

ranking_path = outputs_dir / 'ranking_probabilidades.csv'
ranking_probabilidades.to_csv(ranking_path, index=False, encoding='utf-8-sig')

# Feature Importance: mostra quais colunas tiveram mais peso no modelo.
if hasattr(modelo, 'feature_importances_'):
    importancias = modelo.feature_importances_
elif hasattr(modelo, 'coef_'):
    importancias = abs(modelo.coef_).ravel()
else:
    raise AttributeError('O modelo usado não possui feature_importances_ nem coef_.')

feature_importance = pd.DataFrame({
    'variavel': X_train.columns,
    'importancia': importancias
})

feature_importance = (
    feature_importance
    .sort_values('importancia', ascending=False)
    .reset_index(drop=True)
)

feature_importance_path = outputs_dir / 'feature_importance.csv'
feature_importance.to_csv(feature_importance_path, index=False, encoding='utf-8-sig')

print(f"Previsões detalhadas salvas em: {previsoes_detalhadas_path}")
print(f"Ranking salvo em: {ranking_path}")
print(f"Feature importance salvo em: {feature_importance_path}")

print("\nTop 10 seleções por probabilidade:")
display(ranking_probabilidades.head(10))

print("\nTop 10 variáveis mais importantes:")
display(feature_importance.head(10))

Previsões detalhadas salvas em: c:\Users\gusta\Desktop\dende-softhouse-modelo-preditivo-world-cup-2026-equipe-lauro\outputs\previsoes_detalhadas_test.csv
Ranking salvo em: c:\Users\gusta\Desktop\dende-softhouse-modelo-preditivo-world-cup-2026-equipe-lauro\outputs\ranking_probabilidades.csv
Feature importance salvo em: c:\Users\gusta\Desktop\dende-softhouse-modelo-preditivo-world-cup-2026-equipe-lauro\outputs\feature_importance.csv

Top 10 seleções por probabilidade:


,posicao,team_name,country_code,confederation,probabilidade_campeao,qtd_amostras,probabilidade_campeao_pct
0,1,Argentina,ARG,CONMEBOL,0.752500,12,75.25
1,2,France,FRA,UEFA,0.748750,16,74.88
2,3,England,ENG,UEFA,0.730833,12,73.08
3,4,Spain,ESP,UEFA,0.722500,8,72.25
4,5,Brazil,BRA,CONMEBOL,0.714545,11,71.45
5,6,USA,USA,CONCACAF,0.601111,9,60.11
6,7,Croatia,CRO,UEFA,0.573750,8,57.38
7,8,Netherlands,NED,UEFA,0.563333,6,56.33
8,9,Germany,DEU,UEFA,0.557500,8,55.75
9,10,Italy,ITA,UEFA,0.532857,7,53.29



Top 10 variáveis mais importantes:


,variavel,importancia
0,avg_player_rating,0.081009
1,win_rate_last_year,0.066595
2,goals_scored_avg,0.060559
3,passing_accuracy,0.059107
4,experience_avg_caps,0.059065
5,market_value_million_eur,0.057651
6,fifa_rank,0.057568
7,possession_avg,0.055727
8,fifa_points,0.055329
9,recent_form_score,0.055244
